# RAG Query Engine and Persistent Chatbot

This notebook is now organized like a small RAG application instead of a pile of helper functions. The same behavior is kept, but the responsibilities are grouped into three modules:

1. `ChromaKnowledgeBase`: reconnects to ChromaDB, rebuilds the LlamaIndex vector index, rebuilds BM25, and creates the hybrid retriever.
2. `JsonChatSessionStore`: saves and loads chat memory as one JSON file per chat in the `session` folder.
3. `RagNewsChatbot`: exposes the product-facing actions: ask one question, open a chat, continue a chat, inspect history, and show sources.

```mermaid
flowchart LR
    U[User question] --> APP[RagNewsChatbot]
    APP --> HR[HybridRetriever]
    HR --> VS[Semantic search\nChroma vector index]
    HR --> BM[Keyword search\nBM25]
    VS --> RRF[Reciprocal rank fusion]
    BM --> RRF
    RRF --> LLM[Gemma 3 1B via Ollama]
    LLM --> A[Answer + sources]
    APP <--> MEM[ChatMemoryBuffer]
    MEM <--> JSON[session/*.json]
```

The important product idea is separation of concerns: retrieval, generation, and persistence are related, but they should not be tangled together.

In [ ]:
# Run this once if the LlamaIndex integrations are missing, then restart the kernel.
%pip install llama-index-vector-stores-chroma llama-index-retrievers-bm25 llama-index-llms-ollama

In [1]:
import json
import re
from collections import defaultdict
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import chromadb
from chromadb.errors import NotFoundError
from llama_index.core import QueryBundle, Settings, VectorStoreIndex
from llama_index.core.chat_engine import ContextChatEngine
from llama_index.core.llms import ChatMessage, MessageRole
from llama_index.core.memory import ChatMemoryBuffer
from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.core.retrievers import BaseRetriever
from llama_index.core.schema import NodeWithScore, TextNode
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.llms.ollama import Ollama
from llama_index.retrievers.bm25 import BM25Retriever
from llama_index.vector_stores.chroma import ChromaVectorStore

resource module not available on Windows


In [2]:
PROJECT_ROOT = Path(r"C:\Program Files\Studying\coding\RAG_project")
NEWS_DIR = PROJECT_ROOT / "data" / "hk_free_press_news"
CHROMA_DIR = PROJECT_ROOT / "chromadb_store"
SESSION_DIR = PROJECT_ROOT / "session"
COLLECTION_NAME = "news_chat"

CHUNK_SIZE = 800
CHUNK_OVERLAP = 120
EMBED_MODEL_NAME = "BAAI/bge-small-en-v1.5"
OLLAMA_MODEL = "gemma3:1b"

if not NEWS_DIR.exists():
    raise FileNotFoundError(f"News folder not found: {NEWS_DIR}")

# Store one JSON file per chat session in this folder.
SESSION_DIR.mkdir(parents=True, exist_ok=True)

# LlamaIndex uses Settings as global defaults for retrieval and generation.
Settings.embed_model = HuggingFaceEmbedding(model_name=EMBED_MODEL_NAME)
Settings.llm = Ollama(model=OLLAMA_MODEL, request_timeout=120.0)

print(f"News source folder: {NEWS_DIR}")
print(f"ChromaDB folder: {CHROMA_DIR}")
print(f"Session folder: {SESSION_DIR}")
print(f"Collection name: {COLLECTION_NAME}")
print(f"Embedding model: {EMBED_MODEL_NAME}")
print(f"LLM model: {OLLAMA_MODEL}")

News source folder: C:\Program Files\Studying\coding\RAG_project\data\hk_free_press_news
ChromaDB folder: C:\Program Files\Studying\coding\RAG_project\chromadb_store
Session folder: C:\Program Files\Studying\coding\RAG_project\session
Collection name: news_chat
Embedding model: BAAI/bge-small-en-v1.5
LLM model: gemma3:1b


In [3]:
def reciprocal_rank_fusion(
    semantic_results: list[NodeWithScore],
    keyword_results: list[NodeWithScore],
    top_k: int,
    rrf_k: int = 60,
) -> list[NodeWithScore]:
    """Fuse semantic and keyword result lists using reciprocal rank fusion."""
    # RRF uses rank positions instead of raw scores, so semantic and BM25 scores do not need calibration.
    fused_scores: dict[str, float] = defaultdict(float)
    node_lookup: dict[str, NodeWithScore] = {}

    for results in (semantic_results, keyword_results):
        for rank, result in enumerate(results, start=1):
            node_id = result.node.node_id
            fused_scores[node_id] += 1.0 / (rrf_k + rank)

            # Keep one full node object per id so final results still include text and metadata.
            if node_id not in node_lookup:
                node_lookup[node_id] = result

    ranked_nodes = sorted(
        node_lookup.values(),
        key=lambda result: fused_scores[result.node.node_id],
        reverse=True,
    )

    return [
        NodeWithScore(node=result.node, score=fused_scores[result.node.node_id])
        for result in ranked_nodes[:top_k]
    ]


class HybridRetriever(BaseRetriever):
    """LlamaIndex retriever that combines semantic search, BM25, and RRF."""

    def __init__(
        self,
        semantic_retriever,
        keyword_retriever: BM25Retriever,
        final_top_k: int = 5,
        rrf_k: int = 60,
    ):
        # Keep retrievers injected rather than built inside _retrieve, which makes retrieval reusable and testable.
        self._semantic_retriever = semantic_retriever
        self._keyword_retriever = keyword_retriever
        self._final_top_k = final_top_k
        self._rrf_k = rrf_k
        super().__init__()

    def _retrieve(self, query_bundle: QueryBundle) -> list[NodeWithScore]:
        """Retrieve candidates from both retrievers and return one fused ranking."""
        # Semantic search catches paraphrases and meaning-level matches.
        semantic_results = self._semantic_retriever.retrieve(query_bundle)

        # BM25 catches exact names, rare phrases, dates, and other keyword-heavy queries.
        keyword_results = self._keyword_retriever.retrieve(query_bundle)

        # RRF gives a stable hybrid ranking without manually weighting two different score systems.
        return reciprocal_rank_fusion(
            semantic_results=semantic_results,
            keyword_results=keyword_results,
            top_k=self._final_top_k,
            rrf_k=self._rrf_k,
        )


class ChromaKnowledgeBase:
    """Small adapter around persisted ChromaDB storage and LlamaIndex retrieval."""

    def __init__(self, chroma_dir: Path, collection_name: str):
        self.chroma_dir = chroma_dir
        self.collection_name = collection_name
        self._client = None
        self._collection = None
        self._index = None
        self._nodes: list[TextNode] | None = None
        self._bm25_cache: dict[int, BM25Retriever] = {}

    @property
    def collection(self):
        """Return the persisted Chroma collection, creating the client lazily."""
        # Lazy loading keeps notebook startup cheap until retrieval is actually needed.
        if self._collection is None:
            self.chroma_dir.mkdir(parents=True, exist_ok=True)
            self._client = chromadb.PersistentClient(path=str(self.chroma_dir))
            self._collection = self._client.get_or_create_collection(self.collection_name)
        return self._collection

    @property
    def index(self) -> VectorStoreIndex:
        """Reconnect LlamaIndex to the existing Chroma vector store."""
        # The vector index is reconstructed from Chroma; we do not reinsert or re-embed documents here.
        if self._index is None:
            vector_store = ChromaVectorStore(chroma_collection=self.collection)
            self._index = VectorStoreIndex.from_vector_store(
                vector_store=vector_store,
                embed_model=Settings.embed_model,
            )
        return self._index

    @property
    def nodes(self) -> list[TextNode]:
        """Rebuild TextNode objects from Chroma documents and metadata for BM25."""
        # BM25 needs local text nodes because it is a keyword index, not a vector database query.
        if self._nodes is None:
            records = self.collection.get(include=["documents", "metadatas"])
            documents = records.get("documents", []) or []
            metadatas = records.get("metadatas", []) or []
            ids = records.get("ids", []) or []

            self._nodes = [
                TextNode(id_=node_id, text=document_text, metadata=metadata or {})
                for node_id, document_text, metadata in zip(ids, documents, metadatas)
                if document_text
            ]

            if not self._nodes:
                raise ValueError("No stored text nodes were found in the Chroma collection.")

        return self._nodes

    def bm25_retriever(self, top_k: int) -> BM25Retriever:
        """Build or reuse a BM25 retriever for the requested candidate count."""
        # Cache by top_k so repeated chat turns do not rebuild the keyword index every time.
        if top_k not in self._bm25_cache:
            self._bm25_cache[top_k] = BM25Retriever.from_defaults(
                nodes=self.nodes,
                similarity_top_k=top_k,
            )
        return self._bm25_cache[top_k]

    def hybrid_retriever(
        self,
        final_top_k: int = 5,
        candidate_top_k: int | None = None,
        rrf_k: int = 60,
    ) -> HybridRetriever:
        """Create the retriever used by query and chat engines."""
        # Pull more candidates than final answers so fusion has room to promote cross-matched results.
        candidate_count = candidate_top_k if candidate_top_k is not None else max(final_top_k * 2, 10)
        semantic_retriever = self.index.as_retriever(similarity_top_k=candidate_count)
        keyword_retriever = self.bm25_retriever(top_k=candidate_count)

        return HybridRetriever(
            semantic_retriever=semantic_retriever,
            keyword_retriever=keyword_retriever,
            final_top_k=final_top_k,
            rrf_k=rrf_k,
        )

    def count(self) -> int:
        """Return the number of vectors stored in the active Chroma collection."""
        return self.collection.count()

In [4]:
CHAT_SYSTEM_PROMPT = """
You are a helpful RAG chatbot for an HK Free Press news knowledge base.
Use the retrieved context as your primary evidence.
If the retrieved context does not contain enough information, say that the knowledge base does not have enough evidence.
Use the current chat history to understand follow-up questions, but do not invent facts that are not supported by the retrieved context.
""".strip()


class JsonChatSessionStore:
    """File-backed storage for chat history, using one JSON file per chat."""

    def __init__(self, session_dir: Path):
        self.session_dir = session_dir
        self.session_dir.mkdir(parents=True, exist_ok=True)

    def safe_name(self, chat_id: str) -> str:
        """Convert a human-readable chat id into a storage-safe file stem."""
        # Production systems usually separate display names from storage identifiers.
        safe_name = re.sub(r"[^A-Za-z0-9_.-]+", "_", chat_id.strip()).strip("_")
        return safe_name or "default_chat"

    def path_for(self, chat_id: str) -> Path:
        """Return the JSON file path for one chat session."""
        # One file per chat makes backup, inspection, deletion, and migration straightforward.
        return self.session_dir / f"{self.safe_name(chat_id)}.json"

    def message_to_dict(self, message: ChatMessage) -> dict[str, Any]:
        """Convert a LlamaIndex ChatMessage into JSON data."""
        # Only persist durable message data; retrievers, indexes, and LLM clients are rebuilt from code.
        role = message.role.value if hasattr(message.role, "value") else str(message.role)
        return {
            "role": role,
            "content": message.content or "",
            "additional_kwargs": message.additional_kwargs or {},
        }

    def message_from_dict(self, data: dict[str, Any]) -> ChatMessage:
        """Rebuild a LlamaIndex ChatMessage from saved JSON data."""
        # Convert role strings back into LlamaIndex roles when possible.
        role_text = data.get("role", "user")
        try:
            role = MessageRole(role_text)
        except ValueError:
            role = role_text

        return ChatMessage(
            role=role,
            content=data.get("content", ""),
            additional_kwargs=data.get("additional_kwargs", {}) or {},
        )

    def memory_from_payload(
        self,
        payload: dict[str, Any],
        token_limit: int,
    ) -> ChatMemoryBuffer:
        """Hydrate a ChatMemoryBuffer from saved messages."""
        # The token limit controls how much conversation can be sent back into the model context.
        memory = ChatMemoryBuffer.from_defaults(token_limit=token_limit)
        for message_data in payload.get("messages", []):
            memory.put(self.message_from_dict(message_data))
        return memory

    def payload_from_memory(self, chat_id: str, memory: ChatMemoryBuffer) -> dict[str, Any]:
        """Create the JSON payload for one chat memory."""
        # Metadata makes future migrations/debugging easier when models or collections change.
        return {
            "schema_version": 1,
            "chat_id": chat_id,
            "collection_name": COLLECTION_NAME,
            "embedding_model": EMBED_MODEL_NAME,
            "llm_model": OLLAMA_MODEL,
            "updated_at": datetime.now(timezone.utc).isoformat(),
            "messages": [self.message_to_dict(message) for message in memory.get_all()],
        }

    def save(self, chat_id: str, memory: ChatMemoryBuffer) -> Path:
        """Save a chat memory with an atomic temp-file replace."""
        session_path = self.path_for(chat_id)
        temp_path = session_path.with_suffix(".tmp")
        payload = self.payload_from_memory(chat_id, memory)

        # Atomic replace avoids leaving a half-written JSON file if the process stops mid-write.
        temp_path.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding="utf-8")
        temp_path.replace(session_path)
        return session_path

    def load(self, chat_id: str) -> dict[str, Any]:
        """Load the raw JSON payload for one chat."""
        session_path = self.path_for(chat_id)
        if not session_path.exists():
            raise FileNotFoundError(f"Saved chat session not found: {session_path}")
        return json.loads(session_path.read_text(encoding="utf-8"))

    def exists(self, chat_id: str) -> bool:
        """Return whether a saved chat exists on disk."""
        return self.path_for(chat_id).exists()

    def list(self) -> list[Path]:
        """List all saved chat JSON files."""
        session_files = sorted(self.session_dir.glob("*.json"))
        for index, session_file in enumerate(session_files, start=1):
            print(f"{index}. {session_file.name}")
        return session_files


class RagNewsChatbot:
    """Facade that exposes the notebook's main RAG product operations."""

    def __init__(
        self,
        knowledge_base: ChromaKnowledgeBase,
        session_store: JsonChatSessionStore,
        final_top_k: int = 5,
        memory_token_limit: int = 3000,
        llm=None,
    ):
        self.knowledge_base = knowledge_base
        self.session_store = session_store
        self.memory_token_limit = memory_token_limit
        self.llm = llm if llm is not None else Settings.llm

        # Build shared retrieval once. QueryEngine and ChatEngine use the same hybrid retriever.
        self.hybrid_retriever = knowledge_base.hybrid_retriever(final_top_k=final_top_k)
        self.query_engine = RetrieverQueryEngine.from_args(
            retriever=self.hybrid_retriever,
            llm=self.llm,
        )
        self.chat_sessions: dict[str, dict[str, Any]] = {}

    def ask(self, question: str):
        """Run single-turn RAG with no chat memory."""
        # Use this for independent questions where the user is not relying on prior turns.
        response = self.query_engine.query(question)
        print(response)
        return response

    def open_chat(
        self,
        chat_id: str,
        load_existing: bool = True,
        overwrite: bool = False,
    ) -> str:
        """Create a new chat or reload an existing chat from disk."""
        # Loading by default gives normal chatbot behavior: reopening a chat restores history.
        if load_existing and self.session_store.exists(chat_id) and not overwrite:
            payload = self.session_store.load(chat_id)
            memory = self.session_store.memory_from_payload(
                payload,
                token_limit=self.memory_token_limit,
            )
        else:
            memory = ChatMemoryBuffer.from_defaults(token_limit=self.memory_token_limit)

        chat_engine = ContextChatEngine.from_defaults(
            retriever=self.hybrid_retriever,
            memory=memory,
            llm=self.llm,
            system_prompt=CHAT_SYSTEM_PROMPT,
        )

        self.chat_sessions[chat_id] = {
            "memory": memory,
            "chat_engine": chat_engine,
            "session_path": self.session_store.path_for(chat_id),
        }

        # Save immediately so a newly created empty chat also has a file on disk.
        self.session_store.save(chat_id, memory)
        return chat_id

    def chat(self, chat_id: str, message: str):
        """Send one message, print the answer, and persist the updated chat."""
        if chat_id not in self.chat_sessions:
            raise KeyError(f"Chat session not opened: {chat_id}. Call open_chat first.")

        # LlamaIndex uses the chat memory plus retrieved context to answer follow-up questions.
        response = self.chat_sessions[chat_id]["chat_engine"].chat(message)

        # Persist after every assistant response so notebook restarts do not lose conversation turns.
        session_path = self.session_store.save(chat_id, self.chat_sessions[chat_id]["memory"])

        print(response.response)
        print(f"\nSaved chat session to: {session_path}")
        return response

    def show_history(self, chat_id: str) -> None:
        """Print the current in-memory history for one opened chat."""
        if chat_id not in self.chat_sessions:
            raise KeyError(f"Chat session not opened: {chat_id}")

        for message in self.chat_sessions[chat_id]["memory"].get_all():
            role = message.role.value if hasattr(message.role, "value") else str(message.role)
            print(f"{role}: {message.content}")
            if role == "assistant":
                print("-" * 40)

    def list_saved_chats(self) -> list[Path]:
        """List saved chat files in the session directory."""
        return self.session_store.list()

    def print_sources(self, response, max_sources: int = 3) -> None:
        """Print source chunks used by a query or chat response."""
        # Source nodes are the easiest way to verify that answers are grounded in retrieved text.
        source_nodes = getattr(response, "source_nodes", []) or []
        for rank, source_node in enumerate(source_nodes[:max_sources], start=1):
            metadata = source_node.node.metadata
            article_date = metadata.get("article_date", "unknown date")
            article_title = metadata.get("article_title", metadata.get("file_name", "unknown article"))
            print(f"\nSource {rank}: {article_date} | {article_title}")
            print(source_node.node.get_content()[:800])

## Build the Application Objects

Run the next cell once per notebook session. It creates:

- `knowledge_base`: reconnects to the persisted `news_chat` ChromaDB collection
- `session_store`: reads and writes chat JSON files under `session/`
- `rag_app`: the single object you use for query, chat, sources, and history

This is the main readability improvement: examples below call methods on `rag_app` instead of manually coordinating many functions and global variables.

## Run the RAG App

The next cells demonstrate the same functionality as before, but through one object: `rag_app`.

Use `rag_app.ask(...)` for one-off questions. Use `rag_app.open_chat(...)` and `rag_app.chat(...)` for multi-turn conversations with persistent memory.

### Initialize the App

In [6]:
# Run this once after reopening the notebook.
# It reconnects to ChromaDB, rebuilds semantic + BM25 retrieval, and prepares chat persistence.
knowledge_base = ChromaKnowledgeBase(CHROMA_DIR, COLLECTION_NAME)
session_store = JsonChatSessionStore(SESSION_DIR)
rag_app = RagNewsChatbot(
    knowledge_base=knowledge_base,
    session_store=session_store,
    final_top_k=5,
)

print(f"Stored vector count: {knowledge_base.count()}")
print("RAG app is ready.")

Stored vector count: 1499
RAG app is ready.


### Single-turn RAG: use this when the question does not need chat history.

In [ ]:
sample_query = "Who is Sanae Takaichi, and what is her relationship with Donald Trump?"
query_response = rag_app.ask(sample_query)

print("-" * 70)
rag_app.print_sources(query_response)

### Multi-turn RAG With Saved Chat History

`rag_app.open_chat(...)` loads the saved JSON file when it exists, or creates a new one when it does not. Every `rag_app.chat(...)` call saves the updated conversation automatically.

In [ ]:
# Create or load one persistent chat session.
# If session/Sanae_Takaichi.json exists, this restores its previous messages.
chat_id = rag_app.open_chat("Sanae Takaichi")

chat_response = rag_app.chat(
    chat_id=chat_id,
    message="Who is Sanae Takaichi?",
)
rag_app.print_sources(chat_response)

In [ ]:
# Follow-up question: this uses the same saved chat memory, so "she" refers to Sanae Takaichi.
follow_up_response = rag_app.chat(
    chat_id=chat_id,
    message="How long did she talk with Donald Trump during a phone call?",
)
rag_app.print_sources(follow_up_response)

In [ ]:
# Inspect the current in-memory history for this chat.
rag_app.show_history(chat_id)

In [ ]:
# List saved chat files on disk.
rag_app.list_saved_chats()

### Reopen an Existing Chat

After reopening the notebook, run the import/config/module cells and the app initialization cell first. Then `rag_app.open_chat("Sanae Takaichi")` restores the saved conversation from `session/Sanae_Takaichi.json`.

In [7]:
loaded_chat_id = rag_app.open_chat("Sanae Takaichi")

In [8]:
rag_app.show_history(loaded_chat_id)

user: Who is Sanae Takaichi?
assistant: According to the provided context, Sanae Takaichi is the Prime Minister of Japan who won a landslide election in snap elections on Sunday. She is viewed as a conservative and security hawk.

Here’s a breakdown of what the context tells us:

*   **Prime Minister:** He is the leader of Japan.
*   **Recent Political Situation:** He was recently criticized by China, leading to a sharp diplomatic backlash.
*   **Policy:** He’s known for accelerating Japan’s defense strategy, including increased military spending and security cooperation with allies.


----------------------------------------
user: How long did she talk with Donald Trump during a phone call?
assistant: The provided context states that Trump and Takaichi “exchanged views mainly on the Indo-Pacific region and confirmed the close cooperation between Japan and the United States” during their phone call.

It doesn’t specify the duration of the call.
----------------------------------------


In [ ]:
# Continue a loaded chat without rerunning the original conversation cells.
reloaded_response = rag_app.chat(
    chat_id=loaded_chat_id,
    message="Why did you describe Sanae Takaichi that way?",
)
rag_app.print_sources(reloaded_response)

NameError: name 'rag_app' is not defined